# rustyplot quickstart

Phase 0 spike: scatter plot, wheel zoom, drag pan, click callback.

Pan and zoom are handled inside the WebAssembly module, so they never touch the kernel.
Only clicks are sent back to Python.

In [ ]:
import pathlib
import sys

# Use the source tree without installing.
sys.path.insert(0, str(pathlib.Path.cwd().parent / "python"))

import numpy as np

import rustyplot as rp

rp.__version__

## A basic scatter plot

Scroll to zoom about the cursor, drag with the left button to pan.
The bottom-right corner of the frame can be dragged to resize it.

In [ ]:
rng = np.random.default_rng(42)
n = 50_000
x = rng.normal(size=n)
y = 0.6 * x + rng.normal(size=n)

fig = rp.scatter(x, y, size=5.0, color="#1f77b4")
fig

The current view syncs back to Python. Pan the figure above, then re-run this cell.

In [ ]:
fig.view  # [x_min, x_max, y_min, y_max]

## Click callbacks

Clicking reports the data coordinates under the cursor, plus the nearest point if one
was hit. Dragging does not count as a click.

In [ ]:
import ipywidgets as ipw

clicks = ipw.Output()


@fig.on_click
def report(event):
    with clicks:
        clicks.clear_output()
        if event["index"] is None:
            print(f"empty space at ({event['x']:.3f}, {event['y']:.3f})")
        else:
            print(
                f"point {event['index']} at "
                f"({event['point_x']:.3f}, {event['point_y']:.3f})"
            )


ipw.VBox([fig, clicks])

## One million points

The Phase 0 performance gate. Zooming and panning should stay smooth.

In [ ]:
n = 1_000_000
x = rng.normal(size=n)
y = 0.6 * x + rng.normal(size=n)

rp.scatter(x, y, size=2.0, color=(0.1, 0.4, 0.8, 0.25), width=800, height=550)

## Per-point size and colour

In [ ]:
n = 2_000
angle = np.linspace(0, 8 * np.pi, n)
radius = np.linspace(0, 1, n)

colors = np.zeros((n, 4), dtype=np.float32)
colors[:, 0] = radius
colors[:, 2] = 1.0 - radius
colors[:, 3] = 0.9

rp.scatter(
    radius * np.cos(angle),
    radius * np.sin(angle),
    size=2 + 18 * radius,
    color=colors,
)